# 28장 · 비동기 병렬 호출 · asyncio.gather로 결과와 예외 받기

asyncio.gather는 여러 비동기 작업을 함께 기다리고, 인자로 넘긴 순서대로 결과를 돌려줍니다. 노트북에서는 셀 안에서 곧바로 await를 쓸 수 있습니다.

## gather로 반환값 두 개 받기

`await asyncio.gather(작업1, 작업2)`는 두 작업을 함께 진행하고 각 반환값을 인자로 적은 순서대로 돌려줍니다. 다음 셀은 API 없이 숫자 두 개를 각각 두 배로 만들어 결과 목록을 확인합니다. Jupyter에서는 코드 셀에 `await`를 바로 쓸 수 있습니다.

In [ ]:
import asyncio

async def double(number):
    await asyncio.sleep(0)
    return number * 2

print(await asyncio.gather(double(3), double(1)))

[6, 2]


결과는 [6, 2]입니다. 첫 칸은 첫 인자인 double(3)의 결과입니다. 뒤의 실습에서는 기다릴 작업을 실제 모델 호출로 바꿉니다.

## 응용 실습 · 독립 검토 동시에 실행하기

## 동시에 시작해도 되는 조건

25장의 초안 작성은 요구 추출 결과를 기다려야 했습니다. 27장의 두 검토자는 같은 초안을 읽고 서로의 출력을 읽지 않습니다. 이 경우 두 요청을 함께 시작할 수 있습니다. 응답 순서를 보고 검토자 이름을 붙이면 빠르게 끝난 쪽에 따라 결과가 뒤바뀔 수 있으므로 요청할 때부터 이름을 함께 둡니다.

## 실습 준비

8장에서 모델을 호출했던 Jupyter 커널을 선택하고, 이 장의 실습 묶음을 풀어 노트북과 pre-mid 폴더를 나란히 둡니다. 새로운 커널이라면 빈 셀에서 `%pip install -U openai`를 실행하고 수업용 API 키를 환경 변수로 설정한 뒤 시작하세요. 키는 노트북 셀이나 제출물에 적지 않습니다. 아래 함수에는 매번 지시와 이번에 읽을 자료를 전달합니다. 출력은 실행할 때마다 표현이 달라질 수 있습니다.

In [ ]:
from pathlib import Path
import os

# ZIP을 푼 폴더에서 노트북을 열면 이 경로에 네 가지 대여 조항이 있습니다.
rules = Path("pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")
model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

async def로 정의한 함수는 네트워크 응답을 기다리는 동안 다른 작업을 진행할 수 있습니다. 함수 안의 await 지점에서 결과가 준비될 때까지 잠시 제어를 넘깁니다. 이 장은 Jupyter 커널의 비동기 실행 환경을 사용합니다.

In [ ]:
import asyncio
from openai import AsyncOpenAI

# 비동기 클라이언트는 네트워크를 기다리는 동안 다른 검토를 진행할 수 있습니다.
async_client = AsyncOpenAI(timeout=45, max_retries=1)
async def read_review(instruction, material):
    result = await async_client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return result.output_text

In [ ]:
draft = '학생증을 친구에게 맡기면 대신 받을 수 있고, 기기는 이틀 뒤까지 가져오면 됩니다.'
requests = {
    "사실": ('규정과 어긋난 초안 문구를 인용하세요. 각 문구에 근거 조항과 원문의 정확한 조건을 붙이세요. 자료로 확인할 수 없는 것은 확인 필요로 적으세요.', rules + "\n" + draft),
    "표현": ('첫 참석자가 안내만 읽고 할 일을 알 수 있는지 검토하세요. 누가, 언제, 어디서 무엇을 할지 모호한 표현 하나와 고치는 방법을 쓰세요. 규정상 사실의 참거짓은 판정하지 마세요.', draft),
}

## asyncio.gather로 성공과 예외 함께 받기

async def로 정의한 함수를 호출하면 실행할 작업이 만들어집니다. await는 그 작업이 끝날 때까지 이 지점에서 결과를 기다립니다. gather는 여러 작업을 함께 시작하고 전달한 순서대로 결과를 돌려줍니다. return_exceptions=True를 쓰면 한 작업에서 생긴 예외도 결과의 해당 자리에 남아 다른 응답을 읽을 수 있습니다.

`one(name)`을 호출하면 아직 답 대신 기다릴 수 있는 코루틴이 생깁니다. 별표 `*`는 작업들을 `gather`의 개별 인자로 풀어 줍니다. `await asyncio.gather(..., return_exceptions=True)`는 이 작업들을 함께 진행하고, 먼저 끝난 순서와 관계없이 **넘긴 순서**로 결과를 돌려줍니다. 실패한 자리에는 예외 객체가 들어갑니다. 그래서 `dict(zip(selected, outcomes))`로 작업 이름을 성공값 또는 예외와 다시 짝지을 수 있습니다.

### 모델을 부르기 전에 예외가 남는 자리 확인하기

In [ ]:
import asyncio

async def check_line(name):
    if name == "표현":
        raise ValueError("빈 초안")
    return "확인"

names = ["사실", "표현"]
values = await asyncio.gather(*(check_line(name) for name in names), return_exceptions=True)
print(values[0])
print(type(values[1]).__name__)

확인
ValueError


“확인”과 ValueError를 한 번에 받았습니다. 두 번째 작업이 실패해도 첫 번째 결과를 쓸 수 있습니다. 아래 실제 요청에서 실패한 작업만 골라 다시 보내는 데 같은 순서가 쓰입니다.

In [ ]:
async def review_batch(selected, fail_name=None):
    async def one(name):
        if name == fail_name:
            # 시험용 오류는 API를 보내기 전에 만들므로 모델 응답과 구분됩니다.
            raise RuntimeError("수업용 실패: API 전송 전")
        instruction, material = requests[name]
        return await read_review(instruction, material)

    outcomes = await asyncio.gather(
        *(one(name) for name in selected), return_exceptions=True
    )
    return dict(zip(selected, outcomes))

In [ ]:
# Jupyter에서는 await를 셀에서 직접 쓸 수 있습니다.
first = await review_batch(["사실", "표현"], fail_name="표현")
for name, value in first.items():
    print(name, type(value).__name__, value)

사실 검토는 실제 SDK 응답이고 표현 검토의 RuntimeError는 교재에서 만든 실패입니다. 모델이 틀린 답을 낸 경우에는 문자열이 돌아옵니다. 그때는 내용 검토가 필요합니다. 전송 실패와 잘못 쓴 응답은 다음 행동이 다릅니다.

### 재시도 결과로 실패한 항목만 교체하기

`dict.update` 또는 딕셔너리 병합으로 다시 받은 값만 교체할 수 있습니다. 다음 셀은 API 호출 없이 합치는 단계만 확인합니다.

In [ ]:
first_run = {"사실": "확인함", "표현": RuntimeError("시간 초과")}
retry = {"표현": "수정 필요"}
merged = {**first_run, **retry}
print(merged["사실"], merged["표현"])

확인함 수정 필요


사실 검토는 그대로 남고 표현 검토만 새 값이 됐습니다. 실제 코드에서는 어떤 역할이 실패했는지 예외 객체로 확인하고 그 역할만 다시 보냅니다.

## 실패한 작업 이름만 gather에 다시 보내기

`return_exceptions=True`로 실행하면 실패한 자리에는 예외 객체가 남습니다. `isinstance(value, Exception)`으로 그 이름만 골라 `review_batch`에 다시 넘깁니다. `{**first, **retried}`는 같은 키의 값을 오른쪽 결과로 갱신하므로 먼저 성공한 호출을 반복할 필요가 없습니다.

In [ ]:
failed_names = [name for name, value in first.items() if isinstance(value, Exception)]
retried = await review_batch(failed_names)
# 같은 이름의 실패 항목만 이번 응답으로 바꿉니다.
combined = {**first, **retried}
print("다시 요청한 검토:", failed_names)
print("앞선 사실 검토를 보존했나?", combined["사실"] == first["사실"])
print(combined["표현"])

failed_names에 표현만 있어야 합니다. 앞선 사실 응답을 새로 만들지 않았다는 증거는 비교값뿐 아니라 두 번째 호출에 들어간 이름입니다. 네트워크 상태 때문에 실제 예외가 더 생겼다면 그 이름과 오류 종류를 기록하고 원인이 해결된 후 다시 실행하세요.

## 표현 검토가 멈췄을 때의 두 갈래

```mermaid
flowchart TD
    A["검토할 초안"] --> B["사실 검토"]
    A --> C["표현 검토"]
    B --> D["이름별 결과 모으기"]
    C --> D
    D --> E["예외가 난 이름 추리기"]
    E --> F["해당 검토만 재요청"]
    D --> G["기존 결과와 합치기"]
    F --> G
```

fail_name이 표현이면 C에서 전송 전에 예외가 납니다. D에는 사실 응답과 표현 예외가 함께 남습니다. E에서 표현 하나만 골라 재요청하고 G에서 그 자리만 갱신합니다. 실패가 없으면 E에서 고른 목록이 비어 있어 새 모델 요청도 없습니다.

## 시도별 실행 내역을 손으로 읽기

**설명용 예시 기록**

| 시도 | 요청한 검토 | 사실 | 표현 |
|---|---|---|---|
| 첫 시도 | 사실·표현 | R2·R3 불일치 지적 | 전송 전 시험 오류 |
| 두 번째 시도 | 표현 | 앞선 응답 보관 | 주체·장소 명시 제안 |

두 번째 줄의 사실 칸이 채워졌지만 새 요청이 있었다는 뜻은 아닙니다. 요청 이름과 저장한 결과를 별개 열로 읽어 보세요. 실제 출력의 문구와 요청 수는 제출한 원문 및 전송 결과로 확인합니다.

## 실습 · 순차 실행과 동시 실행에서 결과 순서 대조하기

사실 대조에 4초, 표현 검토에 3초가 걸린다고 가정합니다. 순서대로 실행하면 두 시간을 더한 약 7초, 함께 시작하면 느린 쪽이 끝날 때까지 약 4초를 기다립니다. 이 수치는 계산 예시입니다. 실제 응답은 네트워크와 입력 길이에 따라 달라집니다. 이제 표현 쪽이 첫 1초에 실패하는 그림을 그려 보세요. 먼저 오류가 났다는 이유로 사실 검토도 실패로 접어야 할까요? 고객에게 보여 줄 부분 결과와 담당자가 다시 시도할 작업을 나눠 적고 아래 코드의 사전 키와 대응시키세요.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 검토 역할을 서로 바꿔 실패시키기

개인 실습 75분. fail_name을 사실로 바꾸고 새 초안에 대해 첫 결과를 받습니다. 요청 전에 예상한 성공 이름, 실제 성공 이름, 다시 요청한 이름을 적으세요. 이어 두 검토가 모두 성공하는 경우에는 다시 부를 API가 없는지 확인합니다. 짝에게 일부러 문제를 만든 줄과 실제 API 호출 줄을 각각 설명하세요.

In [ ]:
selected = ["사실", "표현"]
# 사실 검토만 실패시켜 응답 사전에서 그 자리를 찾습니다.
# review_batch(selected, fail_name="사실")의 반환값에서 Exception인 이름을 모으세요.
# 그 이름만 다시 보내고 첫 번째 표현 검토 결과가 그대로인지 ==로 확인하세요.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>new_draft = &quot;친구 대신 직접 수령하고 빌린 케이블도 함께 반납해 주세요.&quot;
requests = {&quot;사실&quot;: (requests[&quot;사실&quot;][0], rules + &quot;\n&quot; + new_draft),
            &quot;표현&quot;: (requests[&quot;표현&quot;][0], new_draft)}
for problem in [&quot;사실&quot;, None]:
    initial = await review_batch([&quot;사실&quot;, &quot;표현&quot;], fail_name=problem)
    failed = [name for name, value in initial.items() if isinstance(value, Exception)]
    recovered = {**initial, **(await review_batch(failed))}
    print(&quot;실패&quot;, failed, &quot;완료한 이름&quot;, list(recovered))
    print(&quot;기존 표현 검토 보존&quot;, recovered[&quot;표현&quot;] == initial[&quot;표현&quot;])</code></pre>
<p>기대 기록은 실패 이름 [사실], 기존 표현 결과 보존 True입니다. 두 작업이 성공했다면 failed는 빈 목록입니다. 빈 작업 목록에는 모델 호출이 없습니다. 초안이 바뀐 경우에는 두 검토를 새 버전에 다시 실행해야 합니다.</p>
</details>

## 다음 작업에 가져갈 기록

사실·표현 검토 표에 검토 대상 초안과 시도 번호를 남기세요. 다음 장에서는 서로 다른 질문을 나눌 때도 결과 소속을 잃지 않도록 원문과 대응시키겠습니다.

[Python asyncio.gather](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather) · [OpenAI 비동기 클라이언트](https://github.com/openai/openai-python#async-usage)